# 低評価予測モデルの比較と最終選択

## 目的

これまでに作成した特徴量を用いて、低評価予測モデルの最終的な特徴量構成を検討する。

ロジスティック回帰を基準モデルとし、その後Random ForestやBoosting系モデルと比較する。

## 評価方針

- Train：2017年12月まで
- Validation：2018年1月〜4月
- Holdout：2018年5月〜8月
- 特徴量選択・しきい値決定にはValidationを使用する
- Holdoutは最終モデル決定後に一度だけ評価する

## 評価指標

- F1
- Precision
- Recall
- ROC-AUC
- PR-AUC（Average Precision）

低評価レビュー（1〜2点）を陽性クラスとする。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Notebookの作業ディレクトリを確認
current_dir = Path.cwd().resolve()

# プロジェクトルートを取得
if (current_dir / "src" / "project_paths.py").is_file():
    project_root = current_dir
elif (current_dir.parent / "src" / "project_paths.py").is_file():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "プロジェクトルートが見つかりません。"
        "Notebookの実行ディレクトリを確認してください。"
    )

# モジュール検索パスに追加
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 既存の共通パス設定を使用
from src.project_paths import DATA_DIR, PROJECT_ROOT

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)


PROJECT_ROOT: /workspaces/AWS_Python_data_analysis
DATA_DIR: /workspaces/AWS_Python_data_analysis/data


In [6]:
df_model = pd.read_csv(DATA_DIR / "model_improved_6.csv")

print("Shape:", df_model.shape)

print("\n=== 列一覧 ===")
print(df_model.columns.tolist())

print("\n=== order_idの重複数 ===")
print(df_model["order_id"].duplicated().sum())

print("\n=== 欠損数 ===")
print(df_model.isna().sum())

print("\n=== 購入月の範囲 ===")
print(df_model["purchase_month"].min())
print(df_model["purchase_month"].max())


Shape: (95824, 29)

=== 列一覧 ===
['order_id', 'review_score', 'total_price', 'total_freight', 'delivery_delay_days', 'purchase_month', 'customer_state', 'freight_ratio', 'purchase_month_num', 'item_count', 'category_count', 'low_review', 'payment_count', 'payment_type_count', 'main_payment_type', 'max_payment_installments', 'log_freight_ratio', 'month_sin', 'month_cos', 'main_category', 'main_category_gmv', 'customer_unique_id', 'prior_order_count', 'is_repeat_customer', 'prior_customer_gmv', 'prior_avg_order_value', 'days_since_previous_order', 'log_prior_customer_gmv', 'log_prior_avg_order_value']

=== order_idの重複数 ===
0

=== 欠損数 ===
order_id                     0
review_score                 0
total_price                  0
total_freight                0
delivery_delay_days          0
purchase_month               0
customer_state               0
freight_ratio                0
purchase_month_num           0
item_count                   0
category_count               0
low_review      

In [7]:
# Model A：注文・地域・購入時期
numeric_A = [
    "total_price",
    "total_freight",
    "delivery_delay_days",
    "log_freight_ratio",
    "item_count",
    "category_count",
    "month_sin",
    "month_cos",
]

categorical_A = [
    "customer_state",
]

# Model B：支払い特徴量を追加
numeric_B = numeric_A + [
    "payment_count",
    "payment_type_count",
    "max_payment_installments",
]

categorical_B = categorical_A + [
    "main_payment_type",
]

# Model C：商品カテゴリを追加
numeric_C = numeric_B.copy()

categorical_C = categorical_B + [
    "main_category",
]

# Model D：顧客履歴を追加
numeric_D = numeric_C + [
    "prior_order_count",
    "is_repeat_customer",
    "log_prior_customer_gmv",
    "log_prior_avg_order_value",
    "days_since_previous_order",
]

categorical_D = categorical_C.copy()

feature_sets = {
    "A_Order": (numeric_A, categorical_A),
    "B_Payment": (numeric_B, categorical_B),
    "C_Category": (numeric_C, categorical_C),
    "D_History": (numeric_D, categorical_D),
}

for name, (numeric, categorical) in feature_sets.items():
    print(
        name,
        "数値:",
        len(numeric),
        "カテゴリ:",
        len(categorical),
    )


A_Order 数値: 8 カテゴリ: 1
B_Payment 数値: 11 カテゴリ: 2
C_Category 数値: 11 カテゴリ: 3
D_History 数値: 16 カテゴリ: 3


In [8]:
# 時系列でTrain/Validationを分割

train = df_model[df_model["purchase_month"] <= "2017-12"].copy()

validation = df_model[
    (df_model["purchase_month"] >= "2018-01")
    & (df_model["purchase_month"] <= "2018-04")
].copy()

# Holdoutは分割件数のみ確認し、モデル比較では使用しない
holdout_count = (
    (df_model["purchase_month"] >= "2018-05")
    & (df_model["purchase_month"] <= "2018-08")
).sum()

print("Train:", len(train))
print("Validation:", len(validation))
print("Holdout（未使用）:", holdout_count)


Train: 43362
Validation: 27220
Holdout（未使用）: 25242


In [9]:
# モデルA-Dを同じ条件で評価する

from sklearn.metrics import average_precision_score, roc_auc_score

from src import model_metrics, model_training

results = []

for model_name, (numeric, categorical) in feature_sets.items():
    model, y_val, val_prob = model_training.train_logistic_model(
        train,
        validation,
        numeric,
        categorical,
    )

    threshold_results = model_metrics.threshold_table(
        y_val,
        val_prob,
        thresholds=np.arange(0.05, 0.51, 0.01),
    )

    best = threshold_results.sort_values(
        ["f1", "threshold"],
        ascending=[False, True],
    ).iloc[0]

    results.append(
        {
            "model": model_name,
            "threshold": best["threshold"],
            "precision": best["precision"],
            "recall": best["recall"],
            "f1": best["f1"],
            "roc_auc": roc_auc_score(y_val, val_prob),
            "pr_auc": average_precision_score(y_val, val_prob),
        }
    )

comparison = pd.DataFrame(results)

print(
    comparison.sort_values(
        "f1",
        ascending=False,
    ).to_string(index=False)
)

     model  threshold  precision   recall       f1  roc_auc   pr_auc
   A_Order       0.26   0.611806 0.502795 0.551970 0.766418 0.520047
 B_Payment       0.26   0.611656 0.502124 0.551504 0.767047 0.519549
C_Category       0.25   0.566923 0.510396 0.537176 0.770396 0.513088
 D_History       0.25   0.567756 0.508607 0.536557 0.770000 0.512457
